# Introduction to Machine Learning: Linear Regression

Welcome! In this notebook, based on the official [Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course), we will explore **Linear Regression**.

## 📏 What is Linear Regression?

**Linear Regression** is a fundamental Machine Learning algorithm. At its core, it's about finding the **relationship between two variables** by fitting a straight line to the observed data.

- If you have a set of data points (like the size of a house and its price), linear regression tries to draw the "best fitting line" through those points.
- Once you have this line, you can use it to predict the price of a house of any size!

The equation for a line is:
**$$y = wx + b$$**

- **y**: The value we want to predict (e.g., Fare).
- **x**: The input feature we know (e.g., Miles).
- **w (Weight)**: The slope of the line. How much $y$ changes for each unit of $x$.
- **b (Bias)**: The y-intercept. Where the line crosses the y-axis when $x$ is 0.

## 📉 The Loss Function: Measuring our Mistakes

To find the *best* line, the model needs to know how bad its current line is. We measure this "badness" using a **Loss Function**.

### Mean Squared Error (MSE)
The most common loss function for regression is MSE. 
1. It takes the difference between the actual value and the predicted value (the error).
2. It squares that error (to remove negative signs and punish large errors heavily).
3. It calculates the average of these squared errors across all data points.

**Original Formula:**
$$MSE = \frac{1}{N} \sum (y_{actual} - y_{predicted})^2$$


## 🔄 Gradient Descent: Learning from Mistakes

How does the model improve its line? It uses an algorithm called **Gradient Descent**.

Imagine you are blindfolded on a hill and want to find the lowest valley (where loss is 0). You feel the slope of the ground with your feet and take a step downhill. You repeat this until you reach the bottom.

### Deriving the Gradient
To know which way is "downhill," we need the slope (derivative) of the Loss function with respect to our weight ($w$).

1. **Loss Equation for one point**: $Loss = (y_{actual} - y_{predicted})^2$
2. Substitute $y_{predicted}$: $Loss = (y_{actual} - (w \cdot x + b))^2$
3. **Derivative (Chain Rule)**: We take the derivative of the outer square function, then multiply by the derivative of the inner function.
   - Outer derivative: $2 \cdot (y_{actual} - (w \cdot x + b))$
   - Inner derivative (with respect to $w$): $-x$
4. **Derived Gradient Formula**: $Gradient = -2x \cdot (y_{actual} - y_{predicted}) = -2x \cdot Error$

### The Update Rule
We update our weight by taking a small step in the opposite direction of the gradient. The step size is controlled by the **Learning Rate**.

$$w_{new} = w_{old} - (Learning\_Rate \times Gradient)$$

Let's see this in action with code!

In [ ]:
# Manual Gradient Descent Example
weight = 5.0      # Our initial random guess for the slope (Rate per mile)
learning_rate = 0.05 # How big of a step we take down the hill

actual_fare = 25.0
miles = 2.0

print(f"Initial Weight: {weight}")
predicted_fare = weight * miles  # Current prediction: 10.0
print(f"Initial Prediction: ${predicted_fare}")

# 1. Calculate Error
error = predicted_fare - actual_fare  # 10 - 25 = -15

# 2. Calculate the Derived Gradient: -2 * x * Error (ignoring the constant 2 for simplicity in tuning)
# Standard implementations often absorb the '2' into the learning rate, so Gradient = x * Error
gradient = miles * error 

# 3. Update Weight (Move downhill)
weight = weight - (learning_rate * gradient)

new_prediction = weight * miles
print(f"\nNew Weight after 1 step: {weight}")
print(f"New Prediction: ${new_prediction} (Closer to the actual $25!)")

---
## 🛠️ Let's build a real model: Chicago Taxi Fares

We will use real data to predict the fare of a taxi ride based on trip miles.

### Core Concepts to Tune (Hyperparameters)
- **Learning Rate**: The size of the "step" downhill. Too big, you bounce around. Too small, it takes forever.
- **Epochs**: One complete pass over the entire training dataset.
- **Batch Size**: The number of examples the model processes before updating its weights. (Updating after every single example is too slow, waiting for the whole dataset is too memory-intensive, so we use batches).

In [ ]:
# 1. Setup and Import Libraries
# pandas: Used for data manipulation and analysis (like Excel in Python)
import pandas as pd

# tensorflow: Google's powerful Machine Learning library
import tensorflow as tf

# matplotlib: Used for creating plots and charts
import matplotlib.pyplot as plt

# Load the Chicago Taxi dataset
DATA_URL = 'https://download.mlcc.google.com/mledu-datasets/chicago_taxi_train.csv'
df = pd.read_csv(DATA_URL)

# Keep only relevant columns and remove empty rows (dropna)
training_df = df.loc[:, ('TRIP_MILES', 'TRIP_SECONDS', 'FARE')].dropna()
print("Data loaded successfully!")
training_df.head()

### Data Exploration
Let's see some statistics about our data using `pandas.describe()`. This helps us spot anomalies or extreme values.

In [ ]:
training_df.describe()

Is there a relationship between `TRIP_MILES` and `FARE`? Let's check the correlation. A value close to 1 means strong positive correlation.

In [ ]:
print(training_df.corr()['FARE'])

### Defining Model Functions

We will write reusable functions to build and train our models.

**Complex Functions Explained:**
- `tf.keras.Sequential`: Creates a Neural Network where layers are stacked in sequence, one after the other.
- `tf.keras.layers.Dense`: A standard "Dense" layer where every input is connected to the node. A Dense layer with exactly 1 unit is mathematically identical to a Linear Regression model!
- `tf.keras.optimizers.RMSprop`: An advanced version of Gradient Descent. Instead of a fixed learning rate for everything, RMSprop automatically adjusts the learning rate during training to help the model converge faster and more reliably.

In [ ]:
def build_model(learning_rate, feature_count):
    """Builds a Linear Regression model using Keras."""
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(units=1, input_shape=(feature_count,))
    ])
    
    # Compile configures the learning process
    model.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=learning_rate),
        loss='mean_squared_error',
        metrics=[tf.keras.metrics.RootMeanSquaredError()]
    )
    return model

def train_model(model, df, feature_names, label_name, epochs, batch_size):
    """Trains the model."""
    history = model.fit(
        x=df[feature_names],
        y=df[label_name],
        batch_size=batch_size,
        epochs=epochs,
        verbose=0 # Hides the spammy output during training
    )
    return history

def plot_loss(history, label):
    """Helper function to plot the loss curve."""
    plt.plot(history.history['root_mean_squared_error'], label=label)
    plt.xlabel('Epoch')
    plt.ylabel('Root Mean Squared Error (RMSE)')
    plt.legend()

## 🧪 Experiment 1: Baseline Model
Let's predict FARE using only TRIP_MILES.

In [ ]:
learning_rate = 0.001
epochs = 20
batch_size = 50

model_1 = build_model(learning_rate, feature_count=1)
history_1 = train_model(model_1, training_df, ['TRIP_MILES'], 'FARE', epochs, batch_size)

plt.figure(figsize=(8,5))
plot_loss(history_1, 'Baseline (Miles)')
plt.title('Experiment 1: Loss Curve')
plt.show()

## 🧪 Experiment 2: Hyperparameter Tuning
What happens if we increase the Learning Rate? Let's try 0.1 instead of 0.001.

In [ ]:
model_2 = build_model(learning_rate=0.1, feature_count=1)
history_2 = train_model(model_2, training_df, ['TRIP_MILES'], 'FARE', epochs, batch_size)

plt.figure(figsize=(8,5))
plot_loss(history_1, 'LR = 0.001 (Slow & Steady)')
plot_loss(history_2, 'LR = 0.1 (Fast convergence)')
plt.title('Experiment 2: Learning Rate Comparison')
plt.show()

## 🧪 Experiment 3: Feature Engineering
Distance isn't the only thing that affects a taxi fare. Time matters too (especially in traffic). Let's convert `TRIP_SECONDS` into `TRIP_MINUTES` and train a model using BOTH features.

In [ ]:
training_df['TRIP_MINUTES'] = training_df['TRIP_SECONDS'] / 60.0
multi_features = ['TRIP_MILES', 'TRIP_MINUTES']

# Since we have 2 features, feature_count=2
model_3 = build_model(learning_rate=0.1, feature_count=len(multi_features))
history_3 = train_model(model_3, training_df, multi_features, 'FARE', epochs, batch_size)

plt.figure(figsize=(8,5))
plot_loss(history_2, 'Miles Only')
plot_loss(history_3, 'Miles + Minutes')
plt.title('Experiment 3: Adding Features')
plt.show()